In [1]:
import requests
from dotenv import load_dotenv, find_dotenv

import warnings
warnings.filterwarnings('ignore')


from dotenv import load_dotenv, find_dotenv
import os

from openai import OpenAI

from autogen import ConversableAgent, AssistantAgent

In [14]:
load_dotenv(find_dotenv())
api_key = os.getenv("OPENAI_API_KEY")
model = "gpt-4o-mini"


In [15]:
llm_config = {"model": model, "temperature": 0.0, "api_key": api_key}


In [16]:
task = """
        Write a concise, engaging article about
       AI Agentic Workflows. Make sure the article is
       within 350 words.
       """

In [17]:
writer = AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise "
    "articles (with title) on given topics. You must polish your "
    "writing based on the feedback you receive and give a refined "
    "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

In [18]:
reply = writer.generate_reply(messages=[{"content": task, "role": "user"}])
print(reply)

**Unlocking Efficiency: The Power of AI Agentic Workflows**

In an era where efficiency and productivity are paramount, AI agentic workflows are emerging as a transformative force in various industries. These workflows leverage artificial intelligence to automate tasks, enhance decision-making, and streamline processes, ultimately allowing organizations to focus on strategic initiatives rather than mundane operations.

At the core of AI agentic workflows is the concept of autonomy. Unlike traditional automation, which merely executes predefined tasks, agentic workflows empower AI systems to make decisions based on real-time data analysis. This capability enables organizations to respond swiftly to changing circumstances, optimize resource allocation, and improve overall performance.

One of the most significant advantages of AI agentic workflows is their ability to learn and adapt. Machine learning algorithms analyze historical data to identify patterns and predict outcomes, allowing A

In [19]:
critic = AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
    "the writer and provide constructive "
    "feedback to help improve the quality of the content.",
)


In [20]:
res = critic.initiate_chat(
    recipient=writer, message=task, max_turns=3, summary_method="last_msg"
)

Critic (to Writer):


        Write a concise, engaging article about
       AI Agentic Workflows. Make sure the article is
       within 350 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

**Unlocking Efficiency: The Power of AI Agentic Workflows**

In an era where efficiency and productivity are paramount, AI agentic workflows are revolutionizing how organizations operate. These intelligent systems leverage artificial intelligence to automate and optimize complex processes, allowing businesses to focus on strategic initiatives rather than mundane tasks.

At the core of AI agentic workflows is the concept of autonomy. Unlike traditional automation, which merely executes predefined tasks, agentic workflows are designed to make decisions based on real-time data and contextual understanding. This capability enables them to adapt to changing circumstances, ensuring that workflows remain efficient and relevant.

One of t

In [21]:
# === Add a SEO reviewer agent to suggest SEO improvements ===
SEO_reviewer = AssistantAgent(
    name="SEO-Reviewer",
    llm_config=llm_config,
    system_message="You are an SEO reviewer, known for "
    "your ability to optimize content for search engines, "
    "ensuring that it ranks well and attracts organic traffic. "
    "Make sure your suggestion is concise (within 3 bullet points), "
    "concrete and to the point. "
    "Begin the review by stating your role, like 'SEO Reviewer:'.",
)

In [22]:

# == Add a compliance reviewer agent to suggest compliance improvements ==
compliance_reviewer = AssistantAgent(
    name="Compliance-Reviewer",
    llm_config=llm_config,
    system_message="You are a compliance reviewer. You ensure that the content "
    "adheres to the guidelines and regulations of the industry and Google algorithms. "
    "Begin the review by stating your role, like 'Compliance Reviewer:'.",
)

# == Meta-reviewer agent to provide a final review of the content ==
meta_reviewer = AssistantAgent(
    name="Meta-Reviewer",
    llm_config=llm_config,
    system_message="You are a meta-reviewer. You provide a final review of the content, "
    "ensuring that all the feedback from the previous reviewers has been incorporated. "
    "Begin the review by stating your role, like 'Meta Reviewer:'.",
)

In [23]:
# == Orchestrate the conversation between the agents and nested chats to solve the task ==
def reflection_message(recipient, messages, sender, config):
    return f"""Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}"""